<a href="https://colab.research.google.com/github/drod75/nyc_taxi_analysis/blob/main/notebooks/data_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Engineering

The point of this notebook is to use our data we collected to create new data. In particular we will take our taxi trip data and turn it into trip metric data, showcasing metrics for each unique trip within the data, and unique statistics per each day of the summer.

## Setup

In [1]:
# importing libraries, pandas for data manipulation, requests for data gathering, colab for environment variables
import pandas as pd
import geopandas
from google.colab import userdata
from google.colab import drive

In [2]:
# mounting drive for future use
drive.mount('drive')

Mounted at drive


In [3]:
# importing datasets from Google Drive
taxi_df = pd.read_csv("/content/drive/MyDrive/taxi_summer_data.csv")
zone_df = geopandas.read_file("/content/drive/MyDrive/taxi_zones_data.json")

In [4]:
taxi_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10359634 entries, 0 to 10359633
Data columns (total 19 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   vendorid               int64  
 1   tpep_pickup_datetime   object 
 2   tpep_dropoff_datetime  object 
 3   passenger_count        float64
 4   trip_distance          float64
 5   ratecodeid             float64
 6   pulocationid           int64  
 7   dolocationid           int64  
 8   payment_type           int64  
 9   fare_amount            float64
 10  extra                  float64
 11  mta_tax                float64
 12  tip_amount             float64
 13  tolls_amount           float64
 14  improvement_surcharge  float64
 15  total_amount           float64
 16  congestion_surcharge   float64
 17  airport_fee            float64
 18  cbd_congestion_fee     float64
dtypes: float64(13), int64(4), object(2)
memory usage: 1.5+ GB


In [5]:
zone_df.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 263 entries, 0 to 262
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   shape_leng  263 non-null    float64 
 1   shape_area  263 non-null    float64 
 2   zone        263 non-null    object  
 3   locationid  263 non-null    int32   
 4   borough     263 non-null    object  
 5   geometry    263 non-null    geometry
dtypes: float64(2), geometry(1), int32(1), object(2)
memory usage: 11.4+ KB


## Individual Trips Statistics

This branch of our summer data will condense every unique trip and gather there aggregrated statistics, such as fare data, trip length data, if it was in the borough, etc.

In [6]:
# displaying information about the taxi dataframe
taxi_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10359634 entries, 0 to 10359633
Data columns (total 19 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   vendorid               int64  
 1   tpep_pickup_datetime   object 
 2   tpep_dropoff_datetime  object 
 3   passenger_count        float64
 4   trip_distance          float64
 5   ratecodeid             float64
 6   pulocationid           int64  
 7   dolocationid           int64  
 8   payment_type           int64  
 9   fare_amount            float64
 10  extra                  float64
 11  mta_tax                float64
 12  tip_amount             float64
 13  tolls_amount           float64
 14  improvement_surcharge  float64
 15  total_amount           float64
 16  congestion_surcharge   float64
 17  airport_fee            float64
 18  cbd_congestion_fee     float64
dtypes: float64(13), int64(4), object(2)
memory usage: 1.5+ GB


In [7]:
# grouping taxi data by pickup and dropoff locations to calculate metrics
trip_df = taxi_df.groupby(['pulocationid', 'dolocationid'])['dolocationid'].count()
trip_df = trip_df.reset_index(name='trip_count')
trip_df.head()

,pulocationid,dolocationid,trip_count
0,2,76,1
1,2,82,1
2,2,117,1
3,2,122,1
4,2,141,1


In [8]:
# sorting trip data by pickup and dropoff location IDs
trip_df.sort_values(by=['pulocationid','dolocationid'], ascending=True, inplace=True)
trip_df.head(10)

,pulocationid,dolocationid,trip_count
0,2,76,1
1,2,82,1
2,2,117,1
3,2,122,1
4,2,141,1
5,3,3,26
6,3,7,2
7,3,16,1
8,3,18,25
9,3,20,10


In [9]:
# merging trip data with zone borough data based on pickup location ID
trip_df = pd.merge(trip_df, zone_df, left_on='pulocationid', right_on='locationid')
trip_df.info()
trip_df.drop(columns=['geometry', 'shape_leng', 'shape_area', 'locationid','trip_count'], inplace=True)
trip_df.rename(columns={"zone": "puzone", "borough": "puborough"}, inplace=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32819 entries, 0 to 32818
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   pulocationid  32819 non-null  int64   
 1   dolocationid  32819 non-null  int64   
 2   trip_count    32819 non-null  int64   
 3   shape_leng    32819 non-null  float64 
 4   shape_area    32819 non-null  float64 
 5   zone          32819 non-null  object  
 6   locationid    32819 non-null  int32   
 7   borough       32819 non-null  object  
 8   geometry      32819 non-null  geometry
dtypes: float64(2), geometry(1), int32(1), int64(3), object(2)
memory usage: 2.1+ MB


In [10]:
# merging trip data with zone data based on dropoff location ID
trip_df = pd.merge(trip_df, zone_df, left_on='dolocationid', right_on='locationid')
trip_df.drop(columns=['geometry', 'shape_leng', 'shape_area', 'locationid'], inplace=True)
trip_df.rename(columns={"zone": "dozone", "borough": "doborough"}, inplace=True)
trip_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33006 entries, 0 to 33005
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   pulocationid  33006 non-null  int64 
 1   dolocationid  33006 non-null  int64 
 2   puzone        33006 non-null  object
 3   puborough     33006 non-null  object
 4   dozone        33006 non-null  object
 5   doborough     33006 non-null  object
dtypes: int64(2), object(4)
memory usage: 1.5+ MB


In [11]:
# checking if the pickup and dropoff boroughs are the same
trip_df['interborough'] = (trip_df['puborough'] == trip_df['doborough'])

In [12]:
# displaying the first few rows of the merged trip dataframe
trip_df.head()

,pulocationid,dolocationid,puzone,puborough,dozone,doborough,interborough
0,2,76,Jamaica Bay,Queens,East New York,Brooklyn,False
1,2,82,Jamaica Bay,Queens,Elmhurst,Queens,True
2,2,117,Jamaica Bay,Queens,Hammels/Arverne,Queens,True
3,2,122,Jamaica Bay,Queens,Hollis,Queens,True
4,2,141,Jamaica Bay,Queens,Lenox Hill West,Manhattan,False


## Export

In [13]:
# exporting datasets to google drive
trip_df.to_csv('/content/drive/MyDrive/trip_data.csv', index=False)